# 🎙️ VoiceBatch Studio v2.1.2 - [Expressions + GitHub Sync]
अब आप [laugh], [sigh], [cough] का इस्तेमाल कर सकते हैं और फाइलें सीधे GitHub पर सेव होंगी।

In [ ]:
# @title 🔑 Step 1: GitHub कनेक्शन (अपना डेटा भरें)
import os
GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "" # @param {type:"string"}

if GITHUB_USER and GITHUB_TOKEN and REPO_NAME:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    
    %cd {REPO_NAME}
    os.makedirs("outputs", exist_ok=True)
    !pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
    print(f"✅ {REPO_NAME} लिंक हो गया है!")
else:
    print("⚠️ भाई, पहले टोकन और यूजरनेम भरें!")

In [ ]:
# @title 🚀 Step 2: app.py (No-Hakhlaahat Logic)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os, re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def voice_master(text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    
    # सिम्बल्स पर हकलाहट रोकने के लिए क्लीनिंग
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    
    temp_wav = 'outputs/temp_gen.wav'
    
    # XTTS v2 Emotional Synthesis
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=temp_wav, 
        split_sentences=True
    )
    
    y, sr = librosa.load(temp_wav)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    final_path = 'outputs/v_batch_final.wav'
    sf.write(final_path, y, sr)
    return final_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ Professional Emotional Voice Studio')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script (Tags: [laugh], [sigh], [cough])', lines=8)
            smp = gr.Audio(label='Upload Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en'], label='Language', value='hi')
            spd = gr.Slider(0.8, 1.2, 1.0, label='Speed')
            ptc = gr.Slider(-3, 3, 0, label='Pitch')
            sil = gr.Checkbox(label='Silence Remover', value=True)
            btn = gr.Button('Generate Realistic Voice 🚀', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Output Audio')
            gr.Markdown('**टिप:** [sigh] का इस्तेमाल गहरे शब्दों के लिए करें।')

    btn.click(voice_master, [txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py

In [ ]:
# @title ⬆️ Step 3: GitHub पर परमानेंट सेव करें
!git config --global user.email "colab@example.com"
!git config --global user.name "{GITHUB_USER}"
!git add .
!git commit -m "Generated Emotional Audio"
!git push
print("✅ मुबारक हो! फाइलें GitHub पर सेव हो गई हैं।")